# How to use few shot examples

:::info Prerequisites

This guide assumes familiarity with the following concepts:
- [Prompt templates](/docs/concepts/#prompt-templates)
- [Example selectors](/docs/concepts/#example-selectors)
- [LLMs](/docs/concepts/#llms)
- [Vectorstores](/docs/concepts/#vector-stores)

:::

In this guide, we'll learn how to create a simple prompt template that provides the model with example inputs and outputs when generating. Providing the LLM with a few such examples is called few-shotting, and is a simple yet powerful way to guide generation and in some cases drastically improve model performance.

A few-shot prompt template can be constructed from either a set of examples, or from an [Example Selector](https://python.langchain.com/api_reference/core/example_selectors/langchain_core.example_selectors.base.BaseExampleSelector.html) class responsible for choosing a subset of examples from the defined set.

This guide will cover few-shotting with string prompt templates. For a guide on few-shotting with chat messages for chat models, see [here](/docs/how_to/few_shot_examples_chat/).

## Create a formatter for the few-shot examples

Configure a formatter that will format the few-shot examples into a string. This formatter should be a `PromptTemplate` object.

In [1]:
from langchain_core.prompts import PromptTemplate

example_prompt = PromptTemplate.from_template("Question: {question}\n{answer}")

## Creating the example set

Next, we'll create a list of few-shot examples. Each example should be a dictionary representing an example input to the formatter prompt we defined above.

In [2]:
examples = [
    {
        "question": "What the differnt packs of Pepsi?",
        "answer": """
Different packs of Pepsi are Diet Pepsi, 10 - 7.5 FL OZ (222 mL), Cans (Total Vol 75 FL OZ (2.22 L))
""",
    },
    {
        "question": "What are different varities of Lays?",
        "answer": """
Different varities of lays are Bacon Wrapped Jalapeno Popper Flavored Potato Crisps, Barbecue Flavored Potato Crisps
""",
    }
    
]

Let's test the formatting prompt with one of our examples:

In [3]:
print(example_prompt.invoke(examples[0]).to_string())

Question: What the differnt packs of Pepsi?

Different packs of Pepsi are Diet Pepsi, 10 - 7.5 FL OZ (222 mL), Cans (Total Vol 75 FL OZ (2.22 L))



### Pass the examples and formatter to `FewShotPromptTemplate`

Finally, create a [`FewShotPromptTemplate`](https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.few_shot.FewShotPromptTemplate.html) object. This object takes in the few-shot examples and the formatter for the few-shot examples. When this `FewShotPromptTemplate` is formatted, it formats the passed examples using the `example_prompt`, then and adds them to the final prompt before `suffix`:

In [4]:
from langchain_core.prompts import FewShotPromptTemplate

prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="Question: {input}",
    input_variables=["input"],
)

print(
    prompt.invoke({"input": "What are the different flavours of Starbucks Coffee?"}).to_string()
)

Question: What the differnt packs of Pepsi?

Different packs of Pepsi are Diet Pepsi, 10 - 7.5 FL OZ (222 mL), Cans (Total Vol 75 FL OZ (2.22 L))


Question: What are different varities of Lays?

Different varities of lays are Bacon Wrapped Jalapeno Popper Flavored Potato Crisps, Barbecue Flavored Potato Crisps


Question: What are the different flavours of Starbucks Coffee?


By providing the model with examples like this, we can guide the model to a better response.

## Using an example selector

We will reuse the example set and the formatter from the previous section. However, instead of feeding the examples directly into the `FewShotPromptTemplate` object, we will feed them into an implementation of `ExampleSelector` called [`SemanticSimilarityExampleSelector`](https://python.langchain.com/api_reference/core/example_selectors/langchain_core.example_selectors.semantic_similarity.SemanticSimilarityExampleSelector.html) instance. This class selects few-shot examples from the initial set based on their similarity to the input. It uses an embedding model to compute the similarity between the input and the few-shot examples, as well as a vector store to perform the nearest neighbor search.

To show what it looks like, let's initialize an instance and call it in isolation:

In [5]:
from langchain_chroma import Chroma
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_aws import BedrockEmbeddings

example_selector = SemanticSimilarityExampleSelector.from_examples(
    # This is the list of examples available to select from.
    examples,
    # This is the embedding class used to produce embeddings which are used to measure semantic similarity.
    BedrockEmbeddings(model_id="amazon.titan-embed-text-v2:0"),
    # This is the VectorStore class that is used to store the embeddings and do a similarity search over.
    Chroma,
    # This is the number of examples to produce.
    k=1,
)

# Select the most similar example to the input.
question = "What are the different flavours of Starbucks Coffee?"
selected_examples = example_selector.select_examples({"question": question})
print(f"Examples most similar to the input: {question}")
for example in selected_examples:
    print("\n")
    for k, v in example.items():
        print(f"{k}: {v}")

Examples most similar to the input: What are the different flavours of Starbucks Coffee?


answer: 
Different varities of lays are Bacon Wrapped Jalapeno Popper Flavored Potato Crisps, Barbecue Flavored Potato Crisps

question: What are different varities of Lays?
